## **DAY 3 : Advanced Pandas + Feature Engineering**

<br>
<br>


### **Why Feature Engineering is Important**

Feature engineering is often considered the most important part of the machine learning pipeline because:

1.  **Improves Model Accuracy**: Well-crafted features can highlight the underlying patterns in data that a model might otherwise miss.
2.  **Reduces Complexity**: By creating more descriptive features, you can often use simpler, faster models to achieve better results.
3.  **Handles Raw Data Constraints**: It allows models to process non-numeric data (like dates or text) by converting them into meaningful numerical representations.
4.  **Reduces Noise**: It helps in filtering out irrelevant information, focusing the model's 'attention' on what actually drives the prediction target.

<br>
<br>

## **Part 1: The `shift()` Function**

The `shift()` method allows you to shift the index of a Series or DataFrame by a specified number of periods. This is a fundamental technique in feature engineering for creating 'lag' features.

In [11]:
import pandas as pd

# Sample data: Daily sales
data = {'Date': pd.date_range(start='2023-01-01', periods=5),
        'Sales': [100, 120, 110, 150, 140]}
df = pd.DataFrame(data)

# Create a 'Lagged' feature (Yesterday's Sales)
df['Prev_Day_Sales'] = df['Sales'].shift(1)

# Calculate daily change
df['Sales_Change'] = df['Sales'] - df['Prev_Day_Sales']

display(df)

,Date,Sales,Prev_Day_Sales,Sales_Change
0,2023-01-01,100,NaN,NaN
1,2023-01-02,120,100.0,20.0
2,2023-01-03,110,120.0,-10.0
3,2023-01-04,150,110.0,40.0
4,2023-01-05,140,150.0,-10.0


### **Why is `shift()` Important?**

In real-world predictive modeling (especially for time-series), we don't know the future. To predict today's sales, we use data from yesterday, last week, or last month.

**Key Use Cases:**
1.  **Lag Features:** Creating a column for 'Previous Month Sales' to help the model identify seasonality or trends.
2.  **Calculating Velocity/Momentum:** Comparing the current value to a shifted value to see if growth is accelerating or decelerating.
3.  **Target Shifting:** In supervised learning, you often shift the target variable *backwards* to align today's features with tomorrow's outcome for training.

In [12]:
# Example: Creating multiple lags to capture a 2-day trend
df['Lag_1'] = df['Sales'].shift(1)
df['Lag_2'] = df['Sales'].shift(2)

# A simple 'Trend' feature
df['Trend_Direction'] = (df['Sales'] > df['Lag_1']).astype(int)

display(df)

,Date,Sales,Prev_Day_Sales,Sales_Change,Lag_1,Lag_2,Trend_Direction
0,2023-01-01,100,NaN,NaN,NaN,NaN,0
1,2023-01-02,120,100.0,20.0,100.0,NaN,1
2,2023-01-03,110,120.0,-10.0,120.0,100.0,0
3,2023-01-04,150,110.0,40.0,110.0,120.0,1
4,2023-01-05,140,150.0,-10.0,150.0,110.0,0


<br>
<br>

## **Part 2: The `rolling()` Function**

The `rolling()` method provides rolling window calculations. It is commonly used to calculate moving averages, rolling sums, or other statistics over a fixed window of time or observations.

In [13]:
# Calculate a 3-day moving average of sales
# window=3 means it takes the current row and the two previous rows
df['3Day_Moving_Avg'] = df['Sales'].rolling(window=3).mean()


display(df) # (100+120+110)/3 = 110.0

,Date,Sales,Prev_Day_Sales,Sales_Change,Lag_1,Lag_2,Trend_Direction,3Day_Moving_Avg
0,2023-01-01,100,NaN,NaN,NaN,NaN,0,NaN
1,2023-01-02,120,100.0,20.0,100.0,NaN,1,NaN
2,2023-01-03,110,120.0,-10.0,120.0,100.0,0,110.000000
3,2023-01-04,150,110.0,40.0,110.0,120.0,1,126.666667
4,2023-01-05,140,150.0,-10.0,150.0,110.0,0,133.333333


### **Why is `rolling()` Important?**

Rolling windows are essential for dealing with **volatile data**. If you only look at one point in time (like with `shift`), you might be misled by a single outlier.

**Key Use Cases:**
1.  **Smoothing Noise:** A simple moving average (SMA) helps 'smooth out' daily fluctuations so you can see the actual trend.
2.  **Seasonality Analysis:** Rolling sums can tell you the total sales over the last 7 or 30 days, which is more stable than daily sales.
3.  **Volatility Tracking:** You can calculate a rolling standard deviation to see how much the data is 'swinging' over time (often used in financial markets).

In [14]:
# Example: Comparing raw sales to a rolling mean and rolling sum
df['7Day_Sum'] = df['Sales'].rolling(window=7, min_periods=1).sum()
df['Sales_Volatility'] = df['Sales'].rolling(window=3).std()

display(df[['Date', 'Sales', '3Day_Moving_Avg', 'Sales_Volatility']])

,Date,Sales,3Day_Moving_Avg,Sales_Volatility
0,2023-01-01,100,NaN,NaN
1,2023-01-02,120,NaN,NaN
2,2023-01-03,110,110.000000,10.00000
3,2023-01-04,150,126.666667,20.81666
4,2023-01-05,140,133.333333,20.81666


<br>
<br>

## **Part 3: The `diff()` Function**

The `diff()` method calculates the difference of a DataFrame element compared with another element in the DataFrame. It is essentially a shorthand for `df - df.shift(1)` and is frequently used to find the rate of change or to make a non-stationary time series stationary.

In [15]:
# Calculate the first discrete difference of sales
# This is equivalent to Sales - Sales.shift(1)
df['Sales_Diff'] = df['Sales'].diff()

# You can also specify the period
# df['Sales_Diff_2'] = df['Sales'].diff(periods=2)

display(df[['Date', 'Sales', 'Sales_Diff']])

,Date,Sales,Sales_Diff
0,2023-01-01,100,NaN
1,2023-01-02,120,20.0
2,2023-01-03,110,-10.0
3,2023-01-04,150,40.0
4,2023-01-05,140,-10.0


### **Why is `diff()` Important?**

In data science, we often care more about the **change** in a value than the value itself. `diff()` is the standard tool for this.

**Key Use Cases:**
1.  **Making Data Stationary**: Many statistical models (like ARIMA) require data to be 'stationary' (meaning its mean and variance don't change over time). Differencing is the primary way to remove trends and seasonality to achieve this.
2.  **Identifying Momentum**: It helps you see if a metric is growing or shrinking. For example, a positive `Sales_Diff` indicates growth, while a negative one indicates a decline.
3.  **Detecting Outliers**: Sudden, massive spikes in the difference column often point to data errors or significant one-time events that a model needs to account for.



<br>
<br>

## **Part 4: The `pct_change()` Function**

While `diff()` tells you the absolute change (e.g., "We sold 20 more units"), `pct_change()` tells you the **growth rate** (e.g., "Sales grew by 20%"). This scales the change by the previous value, making it easier to compare different datasets.

In [16]:
# Calculate the daily percentage change in sales
df['Sales_Pct_Change'] = df['Sales'].pct_change() * 100

display(df[['Date', 'Sales', 'Sales_Diff', 'Sales_Pct_Change']])

,Date,Sales,Sales_Diff,Sales_Pct_Change
0,2023-01-01,100,NaN,NaN
1,2023-01-02,120,20.0,20.000000
2,2023-01-03,110,-10.0,-8.333333
3,2023-01-04,150,40.0,36.363636
4,2023-01-05,140,-10.0,-6.666667


### **Why is `pct_change()` Important?**

1.  **Normalization**: It allows you to compare the growth of a small store with a large store on equal footing.
2.  **Financial Metrics**: It is the standard way to calculate daily stock returns or inflation rates.
3.  **Threshold Triggers**: It's often used to flag 'unusual' activity, such as a sudden 50% drop in traffic.

<br>
<br>


## **Part 5: The `expanding()` Function**

The `expanding()` method provides expanding transformations. Unlike a rolling window that discards old data, an expanding window includes **all** data from the start of the time series up to the current point.

In [17]:
# Calculate the 'Running Total' (Cumulative Sum) and 'Cumulative Max'
df['Running_Total'] = df['Sales'].expanding().sum()
df['All_Time_High'] = df['Sales'].expanding().max()

display(df[['Date', 'Sales', 'Running_Total', 'All_Time_High']])

,Date,Sales,Running_Total,All_Time_High
0,2023-01-01,100,100.0,100.0
1,2023-01-02,120,220.0,120.0
2,2023-01-03,110,330.0,120.0
3,2023-01-04,150,480.0,150.0
4,2023-01-05,140,620.0,150.0


### **Why is `expanding()` Important?**

1.  **Cumulative Metrics**: It is the standard way to calculate Running Totals, Year-to-Date (YTD) revenue, or cumulative user growth.
2.  **Stability**: While rolling windows show local trends, expanding windows show the 'big picture' evolution of the data.
3.  **Baseline Comparison**: It allows you to compare the current value against the highest value ever seen so far (`expanding().max()`).

<br>
<br>

## **Part 6: The `ewm()` Function**

Exponentially Weighted (EW) functions assign weights that decrease exponentially over time. This means the most recent data point has the highest impact on the average, while older points have less influence.

In [18]:
# Calculate Exponentially Weighted Moving Average (EWMA)
# span=3 is roughly equivalent to a 3-day rolling window, but more reactive
df['EWMA_3Day'] = df['Sales'].ewm(span=3).mean()

display(df[['Date', 'Sales', '3Day_Moving_Avg', 'EWMA_3Day']])

,Date,Sales,3Day_Moving_Avg,EWMA_3Day
0,2023-01-01,100,NaN,100.000000
1,2023-01-02,120,NaN,113.333333
2,2023-01-03,110,110.000000,111.428571
3,2023-01-04,150,126.666667,132.000000
4,2023-01-05,140,133.333333,136.129032


### **Why is `ewm()` Important?**

1.  **Recency Bias**: In fast-moving markets (like stock prices or viral trends), today's data is much more relevant than data from two weeks ago.
2.  **No Lag**: Traditional moving averages often 'lag' behind the current price. EWM reduces this lag, allowing for faster detection of trend reversals.
3.  **Smoothing without Distortion**: It provides a smooth curve like `rolling()`, but stays much closer to the actual data points when sudden changes occur.

<br>
<br>


<br>


## **Part 7: Core Feature Engineering Reference**

To build a high-performing model, we typically focus on three categories of features. Here is your quick-access guide:

| Technique | Pandas Code | Data Science Goal |
| :--- | :--- | :--- |
| **Lagging** | `.shift(1)` | Captures **History** (What happened yesterday?) |
| **Smoothing** | `.rolling(w).mean()` | Captures **Context** (What is the recent trend?) |
| **Momentum** | `.pct_change()` | Captures **Velocity** (Is growth accelerating?) |
| **Stability** | `.rolling(w).std()` | Captures **Risk** (How volatile is the signal?) |
| **Baselines** | `.expanding().max()` | Captures **Global Records** (Is this an all-time high?) |

---

<br>
<br>

## **Part 8: Datetime Feature Engineering**

Machine Learning models cannot 'read' a timestamp string like `2023-10-27 14:30:00`. To make this data useful, we must decompose it into numerical features that capture recurring patterns.

### **Why it's Important:**
1.  **Seasonality**: Is it Monday (low sales) or Saturday (high sales)?
2.  **Periodicity**: Is it the end of the month (payday)?
3.  **Business Logic**: Is it a holiday or a business hour vs. after-hours?
4.  **Trend**: How many days have passed since the start of our dataset?

### **What does `periods=10` mean?**

When using `pd.date_range`, you usually need to provide at least two out of these three:
1.  **`start`**: Where to begin.
2.  **`end`**: Where to stop.
3.  **`periods`**: How many total steps to take.

In your code, `periods=10` tells Pandas: *"Start on Jan 1st and give me exactly 10 rows of data."* Since the default frequency is **'D' (Calendar Day)**, it generates Jan 1st through Jan 10th.

In [28]:
# Quick Demo: Changing periods to 5
demo_dates = pd.date_range(start="2024-01-01", periods=5)
print(f"Generated {len(demo_dates)} dates:")
print(demo_dates)

Generated 5 dates:
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05'],
              dtype='datetime64[ns]', freq='D')


In [26]:
# Ensure the 'Date' column is in datetime format
df['Date'] = pd.to_datetime(df['Date'])

# 1. Basic Component Extraction
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# 2. Weekday/Weekend Extraction
# .dt.weekday returns 0-6 (Monday-Sunday)
df['Day_of_Week'] = df['Date'].dt.weekday
df['Is_Weekend'] = df['Date'].dt.weekday.isin([5, 6]).astype(int)

# 3. Quarterly/Monthly Progress
df['Quarter'] = df['Date'].dt.quarter
df['Is_Month_End'] = df['Date'].dt.is_month_end.astype(int)

# 4. Time-Delta Feature (Days since start)
df['Days_Since_Start'] = (df['Date'] - df['Date'].min()).dt.days

display(df[['Date', 'Day_of_Week', 'Is_Weekend', 'Is_Month_End', 'Days_Since_Start']].head())

,Date,Day_of_Week,Is_Weekend,Is_Month_End,Days_Since_Start
0,2023-01-01,6,1,0,0
1,2023-01-02,0,0,0,1
2,2023-01-03,1,0,0,2
3,2023-01-04,2,0,0,3
4,2023-01-05,3,0,0,4


### **Summary of Datetime Engineering**

By the end of this step, your raw `Date` column has been transformed into a rich set of signals:
*   **Linear Growth**: `Days_Since_Start`
*   **Binary Flags**: `Is_Weekend`, `Is_Month_End`
*   **Cyclical Patterns**: `Month_Sin`, `Month_Cos`

This effectively translates human time into 'Machine Learning Language'. Would you like to try a challenge involving holiday detection or move to Categorical Encoding?

## **PRATICE TASK**

In [38]:
import pandas as pd

data = {
    "Date": pd.date_range(start="2024-01-01", periods=10),
    "Price": [100,105,103,108,110,115,120,118,122,125]
}

df = pd.DataFrame(data)

#LAGGING
df['Lag1'] = df['Price'].shift(1)
df['Lag2'] = df['Price'].shift(2)
df['Lag3'] = df['Price'].shift(3)

#SMOOTHING
df['Rolling_mean_1'] = df['Price'].rolling(window=1).mean()
df['Rolling_mean_2'] = df['Price'].rolling(window=2).mean()

#DIFFERENCE
df['Price_df'] = df['Price'].diff()

#PERCENTAGE_CHANGE
df['Percent_chnage'] = df['Price'].pct_change() * 100

#EXPAND
df['Expander'] = df['Price'].expanding().sum()

#EWM
df['EWM'] = df['Price'].ewm(span=2).mean()

#VOLATILITY
df['Volatility'] = df['Price'].rolling(window=3).std()

#DATE_TIME
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

display(df)


,Date,Price,Lag1,Lag2,Lag3,Rolling_mean_1,Rolling_mean_2,Price_df,Percent_chnage,Expander,EWM,Volatility,Year,Month,Day
0,2024-01-01,100,NaN,NaN,NaN,100.0,NaN,NaN,NaN,100.0,100.000000,NaN,2024,1,1
1,2024-01-02,105,100.0,NaN,NaN,105.0,102.5,5.0,5.000000,205.0,103.750000,NaN,2024,1,2
2,2024-01-03,103,105.0,100.0,NaN,103.0,104.0,-2.0,-1.904762,308.0,103.230769,2.516611,2024,1,3
3,2024-01-04,108,103.0,105.0,100.0,108.0,105.5,5.0,4.854369,416.0,106.450000,2.516611,2024,1,4
4,2024-01-05,110,108.0,103.0,105.0,110.0,109.0,2.0,1.851852,526.0,108.826446,3.605551,2024,1,5
5,2024-01-06,115,110.0,108.0,103.0,115.0,112.5,5.0,4.545455,641.0,112.947802,3.605551,2024,1,6
6,2024-01-07,120,115.0,110.0,108.0,120.0,117.5,5.0,4.347826,761.0,117.651418,5.000000,2024,1,7
7,2024-01-08,118,120.0,115.0,110.0,118.0,119.0,-2.0,-1.666667,879.0,117.883841,2.516611,2024,1,8
8,2024-01-09,122,118.0,120.0,115.0,122.0,120.0,4.0,3.389831,1001.0,120.628087,2.000000,2024,1,9
9,2024-01-10,125,122.0,118.0,120.0,125.0,123.5,3.0,2.459016,1126.0,123.542745,3.511885,2024,1,10
